In [1]:
import sys
from pathlib import Path

current_dir = Path.cwd()
proj_dir = current_dir.parent.parent.resolve() # get project root directory

print(f"Project directory set to: {proj_dir}")

# Add the project root to the system path to import local modules
sys.path.append(str(proj_dir / 'python'))

Project directory set to: D:\dev\dermascan


In [2]:
from pipeline.pipeline_orchestrator import PipelineOrchestrator
from pipeline.dataclasses.pipeline_config import PipelineConfig, BranchConfig, ModelConfig, TrainerConfig, TrainValTestSplitRatio, AugmentationConfig, CustomSplitConfig
from pipeline.dataclasses.enums import ModelType, FilterType, DatasetLoaderType, AugmentationType, SplitType
from pipeline.dataclasses.training_configs import AdamConfig, ReduceLROnPlateauConfig

In [3]:
pipeline_config = PipelineConfig(
    pipeline_id="one_epoch_full_branch_expirement",
    data_path=Path(proj_dir / "data"),
    split_config=CustomSplitConfig(
        train_csv_path=Path(proj_dir / "data" / "custom_split" / "train_df.csv"),
        val_csv_path=Path(proj_dir / "data" / "custom_split" / "val_df.csv"),
        test_csv_path=Path(proj_dir / "data" / "custom_split" / "test_df.csv"),
    ),
    branches=[
        # ──────────────────────────────────────────
        # RGB BRANCHES
        # ──────────────────────────────────────────
        BranchConfig(
            branch_id="rgb_resnet101_frozen_wrs_augements",
            branch_model_config=ModelConfig(
                model=ModelType.RESNET_101,
                filter_type=FilterType.RGB,
                num_classes=2,
                pretrained=True,
                freeze_weights=True,
            ),
            trainer_config=TrainerConfig(
                batch_size=16,
                num_epochs=1,
                learning_rate=0.001,
                optimizer=AdamConfig(),
                scheduler=ReduceLROnPlateauConfig(),
            ),
            dataset_loader=DatasetLoaderType.RGB,
            augmentation=AugmentationConfig(
                train=[
                    AugmentationType.RESIZE,
                    AugmentationType.CENTER_CROP,
                    AugmentationType.RANDOM_HORIZONTAL_FLIP,
                    AugmentationType.RANDOM_VERTICAL_FLIP,
                    AugmentationType.RANDOM_RESIZE_CROP,
                    AugmentationType.COLOR_JITTER,
                ],
                val=[],
            ),
        ),
        # BranchConfig(
        #     branch_id="rgb_resnet50_finetune_wrs",
        #     branch_model_config=ModelConfig(
        #         model=ModelType.RESNET_50,
        #         filter_type=FilterType.RGB,
        #         num_classes=2,
        #         pretrained=True,
        #         freeze_weights=False,
        #     ),
        #     trainer_config=TrainerConfig(
        #         batch_size=16,
        #         num_epochs=20,
        #         learning_rate=0.0001,
        #         optimizer=AdamConfig(),
        #         scheduler=ReduceLROnPlateauConfig(),
        #     ),
        #     dataset_loader=DatasetLoaderType.RGB,
        #     augmentation=AugmentationConfig(
        #         train=[
        #             AugmentationType.RESIZE,
        #             AugmentationType.CENTER_CROP,
        #             AugmentationType.RANDOM_HORIZONTAL_FLIP,
        #             AugmentationType.RANDOM_VERTICAL_FLIP,
        #             AugmentationType.RANDOM_RESIZE_CROP,
        #         ],
        #         val=[],
        #     ),
        # ),
        # ──────────────────────────────────────────
        # GABOR BRANCHES
        # ──────────────────────────────────────────
        # BranchConfig(
        #     branch_id="gabor_resnet50",
        #     branch_model_config=ModelConfig(
        #         model=ModelType.RESNET_50,
        #         filter_type=FilterType.GABOR,
        #         num_classes=2,
        #         pretrained=False,
        #         freeze_weights=False,
        #     ),
        #     trainer_config=TrainerConfig(
        #         batch_size=12,
        #         num_epochs=40,
        #         learning_rate=0.001,
        #         optimizer=AdamConfig(),
        #         scheduler=ReduceLROnPlateauConfig(),
        #     ),
        #     dataset_loader=DatasetLoaderType.FILTER,
        # ),
        # ──────────────────────────────────────────
        # GLCM BRANCHES
        # ──────────────────────────────────────────
        # BranchConfig(
        #     branch_id="glcm_resnet50",
        #     branch_model_config=ModelConfig(
        #         model=ModelType.RESNET_50,
        #         filter_type=FilterType.GLCM,
        #         num_classes=2,
        #         pretrained=False,
        #         freeze_weights=False,
        #     ),
        #     trainer_config=TrainerConfig(
        #         batch_size=16,
        #         num_epochs=40,
        #         learning_rate=0.001,
        #         optimizer=AdamConfig(),
        #         scheduler=ReduceLROnPlateauConfig(),
        #     ),
        #     dataset_loader=DatasetLoaderType.FILTER,
        # ),
        # ──────────────────────────────────────────
        # LBP BRANCHES
        # ──────────────────────────────────────────
        # BranchConfig(
        #     branch_id="lbp_resnet50",
        #     branch_model_config=ModelConfig(
        #         model=ModelType.RESNET_50,
        #         filter_type=FilterType.LBP,
        #         num_classes=2,
        #         pretrained=False,
        #         freeze_weights=False,
        #     ),
        #     trainer_config=TrainerConfig(
        #         batch_size=16,
        #         num_epochs=30,
        #         learning_rate=0.001,
        #         optimizer=AdamConfig(),
        #         scheduler=ReduceLROnPlateauConfig(),
        #     ),
        #     dataset_loader=DatasetLoaderType.FILTER,
        # ),
    ],
)

In [4]:
# Save the pipeline configuration to a JSON file
with open(proj_dir / "python" / "pipeline" / "extensive_branch_expirement_config.json", "w") as f:
    f.write(pipeline_config.model_dump_json(indent=2))

In [5]:
# Load the pipeline configuration from the saved JSON file
json_config = Path(proj_dir / "python" / "pipeline" / "extensive_branch_expirement_config.json").read_text()
pipeline_config = PipelineConfig.model_validate_json(json_config)

In [4]:
pipeline_runner = PipelineOrchestrator(
    pipeline_config=pipeline_config, 
    project_root_path=proj_dir
    )

In [5]:
run = pipeline_runner.run()

Starting pipeline run: one_epoch_full_branch_expirement
Pipeline run path: D:\dev\dermascan\results\one_epoch_full_branch_expirement_2026-05-02_19-16
----------
[BranchRunner] Using device: cuda


Epochs:   0%|          | 0/1 [00:00<?, ?it/s]

Training:   0%|          | 0/367 [00:15<?, ?it/s]

Epoch 1/1 - Train Loss: 0.3111, Train Acc: 0.6333 - Val Loss: 0.1634, Val Acc: 0.8593, Val F1 Macro: 0.4710


2026/05/02 19:18:49 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/05/02 19:18:50 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.11.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/05/02 19:18:59 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.26.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torchvision==0.26.0' without the loca

In [7]:
run

[BranchResult(branch_id='first_test_branch', training_history=TrainingHistory(train_loss=[0.12791629597081564, 0.09487255454008932], val_loss=[0.10416130184212609, 0.09572485798155256], train_accuracy=[0.853382084095064, 0.8583180987202925], val_accuracy=[0.8576300085251491, 0.8576300085251491], val_f1_macro=[0.4616796695731987, 0.4616796695731987]), metrics_result=MetricsResult(f1_macro=0.4616796695731987, f1_weighted=0.791900677903901, accuracy=0.8576300085251491, precision=0.4288150042625746, sensitivity=0.0, specificity=1.0, auc=0.820960464756372, mcc=0.0))]